# Market Model Diagnostics

This notebook diagnoses where specific market models underperform using saved `oof.csv` files.

What it gives you:
- RMSE/MAE/bias by market
- RMSE heatmap by `market x hour`
- Worst-performing market-hour cells (with minimum row filter)
- Tail error diagnostics (per-market upper/lower extremes)
- Optional run-vs-run comparison to see where a new model actually improved


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.metrics import mean_absolute_error, mean_squared_error
from IPython.display import display

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 200)


In [ ]:
RUNS_DIR = Path('../runs')
TRAIN_PATH = Path('../data/train.csv')

# Main run to inspect (must have runs/<RUN_ID>/oof.csv)
RUN_ID = '20260219-1809_c3b064f_per_market_v2'

# Optional comparison run (set to None to skip comparison section)
COMPARE_RUN_ID = None
# COMPARE_RUN_ID = '20260219-1848_fd873d2_per_market_v2_peakexpert'

# Diagnostics controls
MIN_CELL_ROWS = 24
TAIL_UPPER_Q = 0.99
TAIL_LOWER_Q = 0.01


In [ ]:
def resolve_oof_path(run_id: str, runs_dir: Path = RUNS_DIR) -> Path:
    candidates = [
        runs_dir / run_id / 'oof.csv',
        runs_dir / run_id / 'cv_oof.csv',
    ]
    for c in candidates:
        if c.exists():
            return c
    raise FileNotFoundError(
        f'No OOF file found for {run_id}. Expected one of: ' + ', '.join(str(x) for x in candidates)
    )


def list_available_oof_runs(runs_dir: Path) -> list[str]:
    out = []
    for run_dir in sorted(runs_dir.glob('*')):
        if not run_dir.is_dir():
            continue
        if (run_dir / 'oof.csv').exists() or (run_dir / 'cv_oof.csv').exists():
            out.append(run_dir.name)
    return out


available_runs = list_available_oof_runs(RUNS_DIR)
print(f'Found {len(available_runs)} runs with oof.csv/cv_oof.csv')
print('Latest 20:')
for r in available_runs[-20:]:
    print(' -', r)


In [ ]:
def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def load_oof(run_id: str, runs_dir: Path = RUNS_DIR) -> pd.DataFrame:
    p = resolve_oof_path(run_id, runs_dir)
    df = pd.read_csv(p)
    required = {'id', 'market', 'target', 'pred'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'OOF missing columns: {missing}')
    df = df[['id', 'market', 'target', 'pred']].copy()
    df['id'] = df['id'].astype(int)
    return df


def load_train_context(train_path: Path = TRAIN_PATH) -> pd.DataFrame:
    usecols = ['id', 'market', 'delivery_start']
    ctx = pd.read_csv(train_path, usecols=usecols)
    ctx['id'] = ctx['id'].astype(int)
    ctx['delivery_start'] = pd.to_datetime(ctx['delivery_start'])
    ctx['hour'] = ctx['delivery_start'].dt.hour.astype(int)
    ctx['dow'] = ctx['delivery_start'].dt.dayofweek.astype(int)
    ctx['month'] = ctx['delivery_start'].dt.month.astype(int)
    ctx['date'] = ctx['delivery_start'].dt.date
    return ctx


def enrich_oof(oof: pd.DataFrame, train_ctx: pd.DataFrame) -> pd.DataFrame:
    out = oof.merge(
        train_ctx[['id', 'delivery_start', 'hour', 'dow', 'month', 'date']],
        on='id',
        how='left',
        validate='one_to_one',
    )
    if out['delivery_start'].isna().any():
        n_missing = int(out['delivery_start'].isna().sum())
        raise ValueError(f'Missing delivery_start after merge for {n_missing} rows')

    out['error'] = out['pred'] - out['target']
    out['abs_error'] = out['error'].abs()
    out['sq_error'] = out['error'] ** 2
    return out


def market_metrics(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for market, g in df.groupby('market', dropna=False):
        rows.append({
            'market': market,
            'n': int(len(g)),
            'rmse': rmse(g['target'].to_numpy(), g['pred'].to_numpy()),
            'mae': float(mean_absolute_error(g['target'], g['pred'])),
            'bias': float(g['error'].mean()),
            'p95_abs_error': float(g['abs_error'].quantile(0.95)),
        })
    return pd.DataFrame(rows).sort_values('rmse', ascending=False).reset_index(drop=True)


def market_hour_metrics(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (market, hour), g in df.groupby(['market', 'hour'], dropna=False):
        rows.append({
            'market': market,
            'hour': int(hour),
            'n': int(len(g)),
            'rmse': rmse(g['target'].to_numpy(), g['pred'].to_numpy()),
            'mae': float(mean_absolute_error(g['target'], g['pred'])),
            'bias': float(g['error'].mean()),
            'p95_abs_error': float(g['abs_error'].quantile(0.95)),
        })
    return pd.DataFrame(rows).sort_values(['market', 'hour']).reset_index(drop=True)


def tail_metrics(df: pd.DataFrame, q_low: float = 0.01, q_high: float = 0.99) -> pd.DataFrame:
    rows = []
    for market, g in df.groupby('market', dropna=False):
        low = float(g['target'].quantile(q_low))
        high = float(g['target'].quantile(q_high))
        mask_low = g['target'] <= low
        mask_high = g['target'] >= high
        mask_tail = mask_low | mask_high

        g_tail = g.loc[mask_tail]
        g_body = g.loc[~mask_tail]

        rows.append({
            'market': market,
            'q_low': low,
            'q_high': high,
            'tail_n': int(mask_tail.sum()),
            'body_n': int((~mask_tail).sum()),
            'tail_rmse': rmse(g_tail['target'].to_numpy(), g_tail['pred'].to_numpy()) if len(g_tail) > 0 else np.nan,
            'body_rmse': rmse(g_body['target'].to_numpy(), g_body['pred'].to_numpy()) if len(g_body) > 0 else np.nan,
            'tail_mae': float(mean_absolute_error(g_tail['target'], g_tail['pred'])) if len(g_tail) > 0 else np.nan,
            'body_mae': float(mean_absolute_error(g_body['target'], g_body['pred'])) if len(g_body) > 0 else np.nan,
        })

    out = pd.DataFrame(rows)
    out['tail_minus_body_rmse'] = out['tail_rmse'] - out['body_rmse']
    return out.sort_values('tail_minus_body_rmse', ascending=False).reset_index(drop=True)


In [ ]:
train_ctx = load_train_context(TRAIN_PATH)
oof_main = load_oof(RUN_ID, RUNS_DIR)
df_main = enrich_oof(oof_main, train_ctx)

print(f'RUN_ID={RUN_ID}')
print(f'Rows={len(df_main):,}, markets={sorted(df_main.market.unique())}')
print(f'Time range: {df_main.delivery_start.min()} -> {df_main.delivery_start.max()}')
df_main.head()


In [ ]:
overall_rmse = rmse(df_main['target'].to_numpy(), df_main['pred'].to_numpy())
overall_mae = float(mean_absolute_error(df_main['target'], df_main['pred']))
overall_bias = float(df_main['error'].mean())

print(f'Overall RMSE: {overall_rmse:.4f}')
print(f'Overall MAE : {overall_mae:.4f}')
print(f'Overall Bias: {overall_bias:.4f}')

mkt = market_metrics(df_main)
display(mkt)

fig = px.bar(
    mkt.sort_values('rmse', ascending=False),
    x='market',
    y='rmse',
    text='rmse',
    title=f'Market RMSE - {RUN_ID}',
)
fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig.update_layout(yaxis_title='RMSE', xaxis_title='Market')
fig.show()


In [ ]:
mh = market_hour_metrics(df_main)
mh_f = mh[mh['n'] >= MIN_CELL_ROWS].copy()

if mh_f.empty:
    print('No market-hour cells satisfy MIN_CELL_ROWS. Lower MIN_CELL_ROWS and rerun.')
else:
    pivot = mh_f.pivot(index='market', columns='hour', values='rmse').reindex(columns=sorted(mh_f.hour.unique()))
    fig = px.imshow(
        pivot,
        aspect='auto',
        color_continuous_scale='RdYlBu_r',
        title=f'RMSE Heatmap (market x hour), min_rows={MIN_CELL_ROWS} - {RUN_ID}',
    )
    fig.update_layout(xaxis_title='Hour', yaxis_title='Market')
    fig.show()

worst_cells = mh_f.sort_values('rmse', ascending=False).head(30)
print('Worst market-hour cells (filtered):')
display(worst_cells)


In [ ]:
fig = px.box(
    df_main,
    x='market',
    y='error',
    points=False,
    title=f'Residual Distribution by Market - {RUN_ID}',
)
fig.update_layout(yaxis_title='pred - target')
fig.show()

hourly_rmse = (
    df_main.groupby(['market', 'hour'], dropna=False)
    .apply(lambda g: rmse(g['target'].to_numpy(), g['pred'].to_numpy()))
    .rename('rmse')
    .reset_index()
)

fig = px.line(
    hourly_rmse,
    x='hour',
    y='rmse',
    color='market',
    markers=True,
    title=f'Hourly RMSE by Market - {RUN_ID}',
)
fig.update_layout(yaxis_title='RMSE')
fig.show()


In [ ]:
# Market B: OOF prediction overlaid on actual target
market_focus = 'Market B'
mb = df_main[df_main['market'] == market_focus].sort_values('delivery_start').copy()

if mb.empty:
    print(f'No rows found for {market_focus} in {RUN_ID}')
else:
    mb_rmse = rmse(mb['target'].to_numpy(), mb['pred'].to_numpy())
    mb_mae = float(mean_absolute_error(mb['target'], mb['pred']))
    mb_bias = float((mb['pred'] - mb['target']).mean())
    print(
        f"{market_focus}: rows={len(mb):,}, RMSE={mb_rmse:.4f}, MAE={mb_mae:.4f}, bias={mb_bias:.4f}"
    )

    overlay = mb[['delivery_start', 'target', 'pred']].rename(
        columns={'target': 'Actual target', 'pred': 'OOF prediction'}
    )
    overlay_long = overlay.melt(
        id_vars='delivery_start',
        var_name='series',
        value_name='value',
    )

    fig = px.line(
        overlay_long,
        x='delivery_start',
        y='value',
        color='series',
        title=f'{market_focus}: OOF prediction vs actual target over time - {RUN_ID}',
    )
    fig.update_layout(xaxis_title='delivery_start', yaxis_title='target')
    fig.for_each_trace(lambda tr: tr.update(opacity=0.5) if tr.name == 'OOF prediction' else None)
    fig.show()

    worst_mb = (
        mb.assign(abs_error=(mb['pred'] - mb['target']).abs())
        .sort_values('abs_error', ascending=False)
        .head(30)[['id', 'delivery_start', 'target', 'pred', 'error', 'abs_error']]
        .sort_values('delivery_start')
    )
    print('Top 30 absolute errors for Market B:')
    display(worst_mb)




In [ ]:
tail = tail_metrics(df_main, q_low=TAIL_LOWER_Q, q_high=TAIL_UPPER_Q)
display(tail)

fig = px.bar(
    tail.sort_values('tail_minus_body_rmse', ascending=False),
    x='market',
    y='tail_minus_body_rmse',
    text='tail_minus_body_rmse',
    title=f'Tail Penalty by Market (RMSE_tail - RMSE_body), q=({TAIL_LOWER_Q:.2f}, {TAIL_UPPER_Q:.2f})',
)
fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig.update_layout(yaxis_title='Tail minus body RMSE')
fig.show()


In [ ]:
if COMPARE_RUN_ID is None:
    print('Set COMPARE_RUN_ID to compare two runs.')
else:
    oof_cmp = enrich_oof(load_oof(COMPARE_RUN_ID, RUNS_DIR), train_ctx)

    by_market_main = market_metrics(df_main)[['market', 'rmse']].rename(columns={'rmse': 'rmse_main'})
    by_market_cmp = market_metrics(oof_cmp)[['market', 'rmse']].rename(columns={'rmse': 'rmse_compare'})
    market_delta = by_market_main.merge(by_market_cmp, on='market', how='inner')
    market_delta['delta_compare_minus_main'] = market_delta['rmse_compare'] - market_delta['rmse_main']
    market_delta = market_delta.sort_values('delta_compare_minus_main')

    print(f'Main run   : {RUN_ID}')
    print(f'Compare run: {COMPARE_RUN_ID}')
    display(market_delta)

    fig = px.bar(
        market_delta,
        x='market',
        y='delta_compare_minus_main',
        text='delta_compare_minus_main',
        title='RMSE delta by market (compare - main; negative is better)',
    )
    fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
    fig.update_layout(yaxis_title='Delta RMSE')
    fig.show()

    mh_main = market_hour_metrics(df_main)[['market', 'hour', 'rmse']].rename(columns={'rmse': 'rmse_main'})
    mh_cmp = market_hour_metrics(oof_cmp)[['market', 'hour', 'rmse']].rename(columns={'rmse': 'rmse_compare'})
    mh_delta = mh_main.merge(mh_cmp, on=['market', 'hour'], how='inner')
    mh_delta['delta_compare_minus_main'] = mh_delta['rmse_compare'] - mh_delta['rmse_main']

    pivot_delta = mh_delta.pivot(index='market', columns='hour', values='delta_compare_minus_main')
    fig = px.imshow(
        pivot_delta,
        aspect='auto',
        color_continuous_scale='RdBu',
        color_continuous_midpoint=0.0,
        title='Market x hour RMSE delta (compare - main; blue is improvement)',
    )
    fig.update_layout(xaxis_title='Hour', yaxis_title='Market')
    fig.show()


## How to turn diagnostics into model changes

Use the worst `market x hour` cells and tail-penalty table to drive per-market settings:

- Increase hour expert weight only for markets/hours with persistently high RMSE.
- Use `two_sided` tails for markets with large negative errors; keep `upper` where spikes are mostly positive.
- Increase `tail_gate_threshold` for markets with too many false tail triggers.
- Add sample weights for high-error market-hour slices instead of globally upweighting all tails.
- Keep a run-vs-run compare table so every tweak is validated per market, not just on global RMSE.
